# To Dos
- Solve the forme issue regarding stats

# Trying to quantify advantage

Rather than giving the model a bunch of basic information (types, stats, known moves) and hoping that the model learns the stuff we already know (Fire > Grass > Water > Fire), let's tell the model the stuff that we already know.  But how do you feed in the type chart and whatnot into the model?  Seems hard.

Instead, let's bake all of our knowledge into a set of offensive and defensive stats (we'll call these "advantage" stats) that are derived for each specific battle.  Then, we'll replace the basic stats with these "advantage" stats.

We will start off by neglecting the movepool.  We could try to map each move to its category, type, and base power, but that could be a lot of work.  It's something to explore in the future.

## Damage Approximator
$\DeclareMathOperator{\dmg}{dmg}$
$\DeclareMathOperator{\adv}{adv}$
$\DeclareMathOperator{\ttko}{ttko}$
$\newcommand{\M}{\mathrm{M}}$
$\newcommand{\Atk}{\mathrm{Atk}}$
$\newcommand{\Defen}{\mathrm{Def}}$
$\newcommand{\SpAtk}{\mathrm{SpAtk}}$
$\newcommand{\SpDef}{\mathrm{SpDef}}$
The advantage stat starts with an approximation of damage. Given Pokemon $\M_1$ and $\M_2$ from Team 1 and Team 2, respectively, we approximate the <u>expected damage</u> $\dmg(\M_1,\M_2)$, a fraction of $H_2$ (the hit points of $\M_2$), that $\M_1$ does to $\M_2$ by selecting its best STAB move[^1]. Although different moves have different base *Power*s, we set $\mathrm{Power}=80$ for all moves for simplicity.

[^1]: Or a not-very-effective coverage move in the event that is better--this could be updated if we get data suggesting that in the event that a mon's best move is a coverage move, the coverage move is often neutral or better.

In addition, we define the <u>Effective</u> Attacking and Defending Stats $A_i$ and $D_i$ of $\M_i$ as follows: if $i \in \{1,2\}$, let $i'$ be the "complementary" element of $\{1,2\}$, so that $\{i,i'\} = \{1,2\}$. Then
$$ 
    A_i := \max\{\Atk_i, \SpAtk_i\} 
    \qquad\text{and}\qquad
    D_{i'} := \begin{cases}
        \Defen_{i}, & A_{i} = \Atk_i, \\
        \SpDef_{i}, & A_{i} = \SpAtk_{i},
    \end{cases}
$$
and *vice-versa* for $A_{i'}$ and $D_{i}$. Letting $L_i$ and $H_i$ be the Level and HP of $\M_i$, we set

$$
    \dmg(\M_{1},\M_{2}) := \frac{0.925}{H_2} \left(\frac{ 80\left(\tfrac{2}{5} L_{1} + 2\right) \cdot \frac{A_1}{D_2}}{50} + 2\right) \cdot E(\M_{1}, \M_{2}),
$$
where the <u>Effectiveness Multiplier</u>

$$ 
    E(\M_{1}, \M_{2}) := \max\left\{
        \frac{1}{2},\,\,
        1.5 \cdot \max_{T_1 \in \mathrm{Types}(\M_{1})} \mathrm{eff}(T_1, T_2)\mathrm{eff}(T_1, T_2'),
        \right\} 
$$

with $\text{eff}(T_1, T_2)$ being determined by the Type Chart, so that, e.g., 
$$ 
    \text{eff}(\text{water},\text{fire}) = 2, \qquad\text{and}\qquad \text{eff}(\text{fire},\text{water}) = \frac{1}{2}.
$$

Some notes:
- We allow $\dmg$ to exceed 1, as the amount by which it exceeds 1 may actually matter (think: reflect/light screen/aurora veil or resistance berries).
- The definition of $\dmg$ above uses a simplified version of the damage formula found [here](https://bulbapedia.bulbagarden.net/wiki/Damage#Generation_V_onward)
- For simplicity, we have set the base Powers of the moves used to be all be 80, which is the source of that factor in the formula.
- The factor $0.925$ is the mean of a $\mathrm{Unif}(0.85,1)$ random variable.

Notes for $E$:
- $E$ is meant to approximate the product of $\mathrm{STAB}$ and Type.
- The formula for $E$ inherently assumes that $M_1$ is only using STAB moves (this is the factor of $1.5$ present there); this could be updated to account for coverage moves in a future iteration on this stat.
- The $\max\{\frac{1}{2}, \cdot\}$ in $E(\M_1,\M_2)$ is to prevent $E$ from having value 0. It is very rare (though it does happen) that $M_1$ will be unable to damage $M_2$. The factor of $\frac{1}{2}$ is used because that is a multiplier for a "not-very-effective" coverage move. This could be resolved by replacing the maximum over $T_1$ by a maximum over $M_1$'s move types.

More Notes:
- Damage or speed-boosting items are not be accounted for. This could be resolved in an ad-hoc way by checking for common boosting items (choice items, life orb), or resolved in a systemic way using the Smogon damage calculator to replace the offensive advantage stat.
- Damage/stat-modifying abilities like Levitate, Thick Fat, or Sword of Ruin are not accounted for. This could 'only' be resolved by using the Smogon damage calculator to replace this offensive advantage stat.
- Type chart modifying moves like Freeze-Dry are not accounted for.

## Advantage

The only (relevant) thing that doesn't go into the damage approximator is speed. Speed is difficult to incorporate into advantage. There are a few reasons for this:
  1. The only important feature of speed differential (meaning $S_{1} - S_{2}$) is its sign; magnitude is meaningless here, so multiplying $\dmg$ by speed differential would be a bad idea.
  2. The impact of speed differential can be large or small.  If you consider a hypothetical Weavile versus Iron Boulder matchup, each has a super-effective STAB on the other (meaning it has a type-multiplier equal to 3)!  In that situation, Weavile has the advantage because it goes first.  However, if you consider a Weavile versus Swampert matchup (where each has a type-multiplier of 1.5), the Swampert has the advantage in spite of its speed disadvantage due to its overall bulk.  My initial thought is that speed matters a lot when both pokemon are doing about the same amount of damage to one another, but doesn't matter very much when the pokemon are doing very different amounts of damage.  So 'having a speed advantage' should not correspond to a constant factor.

Also worth noting is that advantage depends not just on how much damage you're doing to your opponent, but how much damage your opponent is doing to you!

$\DeclareMathOperator{\Dttko}{\Delta_{\ttko{}}}$
$\newcommand{\Mon}{\mathrm{Mon}}$
Maybe try computing 'turns to KO' for each mon and look at differential.  Let's set
$$
    \ttko(\M_{1},\M_{2}) = \left\lceil \frac{1}{\dmg(\M_{1},\M_{2})} \right\rceil
$$
So we get something like
$$ 
    \Delta_{\ttko{}}(\M_{1},\M_{2}) = \ttko(\M_1,\M_2) - \ttko(\M_1,\M_2).
$$

Here, bigger is better for $\M_{1}$.

Properties that I want for $\adv$:
- There should be a nice relationship between $\adv(\M_1,\M_2)$ and $\adv(\M_2,\M_1)$, ($a+b=1$ with $0 < a,b < 1$?  $ab = 1$?)
- If $\Dttko{} \approx 0$ and both $\ttko \approx 1$, the faster Mon should have a large $\adv$, as the faster Mon just OHKOs the slower Mon with no cost.
- If $\Dttko{} \approx 0$ but both $\ttko \gg 1$, then the faster Mon should one have a small advantage, as here the faster Mon eventually KOs the slower Mon, but both inflict comparable damage on each other.
- If $\Dttko{} \gg 1$, the Mon with the smaller $\ttko$ should have a big advantage, as here one Mon clearly overpowers the other.

$\DeclareMathOperator{\dmgovo}{dmg_{tot}}$
$\DeclareMathOperator{\toko}{toko}$

So maybe $\adv$ should represent something like: expected total damage dealt to opponent in a 1v1 matchup? If we let $n$ denote the round number <u>in which the KO occurs</u>, then the faster Mon gets to go $n$ times and the slower Mon gets to go $n$ or $n-1$ times depending on who wins. 

Then

$$
    \toko(\M_{1},\M_{2}) = \min\Big\{\ttko(\M_{1},M_{2}), \ttko(\M_{2},\M_{1})\Big\}
$$

So

$$
\dmgovo(\M_{1},\M_{2}) =
\begin{cases}
    \dmg(\M_{1},\M_{2})\cdot \big({\toko(\M_{1},\M_{2}) - 1}\big)  &\text{if $S_1 < S_2$ and $\M_2$ KOs $\M_1$},\\
    \dmg(\M_{1},\M_{2})\cdot \toko(\M_{1},\M_{2})         &\text{else.}
\end{cases}
$$

Then we can do something like set 
$$
    \adv(\M_{1},\M_{2}) := \dmgovo(\M_{1},\M_{2})
$$
or we can do something fancy and make it symmetric like 
$$
    \adv(\M_{1},M_{2}) := \dmgovo(\M_{1},\M_{2}) - \dmgovo(\M_{2},\M_{1}).
$$

Regardless, this should be enough to get started.

## Potential problems with advantage stats

Some pokemon are not good because of their stats.  Take Sableye for example.  It has atrocious stats, but can win a match on the strength of its ability, Prankster.  These advantage stats won't account for that.  (On the other hand, neither will training on 12-dimensional info above.)

Other pokemon don't rely on their offensive stats for damage (think Toxapex).

Yet more pokemon rely heavily on priority moves.

# Testing

In order to test the FullPokemon class, we need:

Testing Effectiveness multiplier $E(\M_1,\M_2)$:
  - $\M_{1}$ has a $4\times$ effective STAB ($\M_{1}$ = Weavile, $\M_{2}$ = Salamence)
  - $\M_{1}$ has a $2\times$ effective STAB ($\M_{1}$ = Weavile, $\M_{2}$ = Haxorus)
  - $\M_{1}$'s best STAB is neutral ($1\times$) ($\M_{1}$ = Weavile, $\M_{2}$ = Corviknight)
  - $\M_{1}$'s best STAB is $\frac{1}{2}\times$ effective ($\M_{1}$ = Weavile, $\M_{2}$ = Chien-Pao)
  - $\M_{1}$'s best STAB is $\frac{1}{4}\times$ effective ($\M_{1}$ = Conkeldurr, $\M_{2}$ = Fezandipiti)
  - $\M_{1}$'s best STAB is $0\times$ effective ($\M_{1}$ = Banette, $\M_{2}$ = Wigglytuff)  

Testing $\dmg$:
  - already tested manually by comparing physical and special attackers on the calcs at calc.pokemonshowdown.com

Testing $\dmgovo$:
  - $\M_{1}$ is faster than $\M_{2}$ and KOs $\M_{2}$ ($\M_{1}$ = Weavile, $\M_{2}$ = Salamence)
  - $\M_{2}$ is faster than $\M_{1}$ and KOs $\M_{1}$ ($\M_{1}$ = Sinistcha?, $\M_{2}$ = Weavile)
  - $\M_{1}$ is faster than $\M_{2}$ yet is KOd by $\M_{2}$ ($\M_{1}$ = Weavile, $\M_{2}$ = Conkeldurr)
  - $\M_{2}$ is faster than $\M_{1}$ yet is KOd by $\M_{1}$ ($\M_{1}$ = Swampert, $\M_{2}$ = Corviknight?)
  - $\M_{1}$ and $\M_{2}$ have a speed tie ($\M_{1}$ = Lanturn, $\M_{2}$ = Toxtricity)

In [9]:
import sys, os, json
import pandas as pd
from pathlib import Path

sys.path.insert(1,os.path.abspath('..'))
from tools import battle as b

In [10]:
desired_mon_names = [
    'Weavile', 'Salamence',
    'Haxorus', 'Corviknight',
    'Chien-Pao', 'Conkeldurr',
    'Fezandipiti', 'Banette',
    'Wigglytuff', 'Sinistcha',
    'Swampert', 'Lanturn',
    'Toxtricity'
]

log_dir = Path("../data/replays/gen9-randombattle")

desired_team_dict = {}

for file in log_dir.iterdir():
    if file.name.endswith(".json"):
        with file.open() as battle_json:
            battle = json.load(battle_json)
        for mon in desired_mon_names:
            for team_index in range(2):
                if mon in battle["teams_full"][team_index].keys():
                    desired_team_dict[mon] = battle["teams_full"][team_index][mon]
                    desired_mon_names.remove(mon)
desired_team_dict

{'Lanturn': {'name': 'Lanturn',
  'species': 'Lanturn',
  'speciesId': 'lanturn',
  'gender': 'F',
  'shiny': False,
  'level': 89,
  'moves': ['scald', 'thunderbolt', 'thunderwave', 'icebeam'],
  'ability': 'Volt Absorb',
  'evs': {'hp': 85, 'atk': 0, 'def': 85, 'spa': 85, 'spd': 85, 'spe': 85},
  'ivs': {'hp': 31, 'atk': 0, 'def': 31, 'spa': 31, 'spd': 31, 'spe': 31},
  'item': 'Leftovers',
  'teraType': 'Flying',
  'role': 'Bulky Attacker',
  'bvs': {'hp': 125, 'atk': 58, 'def': 58, 'spa': 76, 'spd': 76, 'spe': 67},
  'stats': {'hp': 367,
   'atk': 108,
   'def': 154,
   'spa': 186,
   'spd': 186,
   'spe': 170},
  'types': ['Water', 'Electric']},
 'Haxorus': {'name': 'Haxorus',
  'species': 'Haxorus',
  'speciesId': 'haxorus',
  'gender': 'F',
  'shiny': False,
  'level': 77,
  'moves': ['dragondance', 'earthquake', 'outrage', 'ironhead'],
  'ability': 'Mold Breaker',
  'evs': {'hp': 85, 'atk': 85, 'def': 85, 'spa': 85, 'spd': 85, 'spe': 85},
  'ivs': {'hp': 31, 'atk': 31, 'def': 3

In [11]:
test_pokemon = {mon : b.FullPokemon(desired_team_dict[mon]) for mon in desired_team_dict.keys()}

In [8]:
test_pokemon['Weavile'].stats

{'hp': 240, 'atk': 235, 'def': 148, 'spa': 117, 'spd': 180, 'spe': 243}

In [24]:
print("Now testing type_multiplier")

def E_test(M1, M2, val) : 
    return None if b.FullPokemon.type_multiplier(test_pokemon[M1],test_pokemon[M2]) == val else f"{M1} v. {M2} failed."

E_test('Weavile', 'Salamence', 6)
E_test('Weavile', 'Haxorus', 3)
E_test('Weavile', 'Corviknight', 1.5)
E_test('Weavile', 'Chien-Pao', 0.75)
E_test('Conkeldurr', 'Fezandipiti', 0.5)
E_test('Banette', 'Wigglytuff', 0.5)

print()
print("Now testing one_v_one_damage")

# ovo = [dmg_ovo(M1,M2), dmg_ovo(M2,M1)]
ovo = b.FullPokemon.one_v_one_damage(test_pokemon['Weavile'],test_pokemon['Salamence'])
assert(ovo[0] >= 1 and ovo[1] == 0), "Weavile and Salamence failed"

ovo = b.FullPokemon.one_v_one_damage(test_pokemon['Sinistcha'],test_pokemon['Weavile'])
assert(0 < ovo[0] < 1 and ovo[1] >= 1), "Sinistcha and Weavile failed"

ovo = b.FullPokemon.one_v_one_damage(test_pokemon['Weavile'],test_pokemon['Conkeldurr'])
assert(0 < ovo[0] < 1 and ovo[1] >=1),"Weavile and Conkeldurr failed"

ovo = b.FullPokemon.one_v_one_damage(test_pokemon['Swampert'],test_pokemon['Corviknight'])
assert(ovo[0] >= 1 and 0 < ovo[1] < 1), "Swampert and Corviknight failed"

ovo = b.FullPokemon.one_v_one_damage(test_pokemon["Lanturn"],test_pokemon["Toxtricity"])
assert(ovo[0] >= 1 and 0.72 <= ovo[1] < 1), "Lanturn and Toxtricity failed"

Now testing type_multiplier

Now testing one_v_one_damage


## Examples

In [25]:
id = "2631906096"
with open("../data/replays/gen9-randombattle/gen9randombattle-" + id + ".json","r") as battle_json:
    data = json.load(battle_json)

team1 = [b.FullPokemon(data["teams_full"][0][key]) for key in data["teams_full"][0].keys()]
team2 = [b.FullPokemon(data["teams_full"][1][key]) for key in data["teams_full"][1].keys()]

col_names = [f"adv_over_{mon2.name}" for mon2 in team2]
col_names.insert(0,"p1_mon")

rows = []
for i,mon1 in enumerate(team1):
    rows.append([b.FullPokemon.advantage(mon1,mon2) for mon2 in team2])
    rows[i].insert(0,mon1.name)
df = pd.DataFrame(rows,columns=col_names)

adv_matrix = [[b.FullPokemon.advantage(mon1,mon2) for mon2 in team2] for mon1 in team1]

sum(sum(adv_matrix[i]) for i in range(len(adv_matrix)))

4.815082462271961

In [72]:
import numpy as np
import numpy.linalg as linalg
import scipy.linalg as alg

# team1 attacking team2
X = np.array([
    [np.round(b.FullPokemon.advantage(M1,M2),2) for M2 in team2] for M1 in team1
])

# team2 attacking team1
Y = np.array([
    [np.round(b.FullPokemon.advantage(M2,M1),2) for M1 in team1] for M2 in team2
])

In [73]:
X

array([[-1.02, -0.12,  0.53,  0.51, -1.48,  0.48],
       [-0.51, -1.26,  1.03,  0.04,  1.19, -0.81],
       [ 0.26,  0.33, -1.24,  0.44, -0.81,  1.13],
       [-0.13,  0.31,  1.42,  0.42,  1.71,  1.08],
       [ 0.32,  0.39,  0.34,  0.49,  0.33,  0.95],
       [-1.37, -0.33, -0.54, -0.18, -0.47, -0.76]])

In [74]:
Y

array([[ 1.02,  0.51, -0.26,  0.13, -0.32,  1.37],
       [ 0.12,  1.26, -0.33, -0.31, -0.39,  0.33],
       [-0.53, -1.03,  1.24, -1.42, -0.34,  0.54],
       [-0.51, -0.04, -0.44, -0.42, -0.49,  0.18],
       [ 1.48, -1.19,  0.81, -1.71, -0.33,  0.47],
       [-0.48,  0.81, -1.13, -1.08, -0.95,  0.76]])

In [26]:
df

,p1_mon,adv_over_Abomasnow,adv_over_Ceruledge,adv_over_Chansey,adv_over_Hippowdon,adv_over_Carbink,adv_over_Klefki
0,Quaquaval,0.715086,1.328708,1.100367,0.649726,-0.168379,-0.605830
1,Pecharunt,1.050588,0.624728,0.492110,-0.674954,-0.312160,0.255685
2,Noivern,-0.756319,0.257875,-0.347247,0.900206,-0.794214,-0.862681
3,Azumarill,-1.031641,-0.066877,-0.567092,0.680872,0.350973,-0.343552
4,Gogoat,-0.768560,-1.037238,1.039033,0.872012,0.853608,-0.258471
5,Klawf,1.121944,1.654554,1.111303,-0.789109,0.170756,-1.030727


## EDA

In [66]:
import sys
import os
import json
import pandas as pd
import matplotlib as plt
import seaborn as sns
from pathlib import Path
sys.path.insert(0,os.path.abspath('..'))
from tools import battle as b
log_dir = Path("../data/replays/gen9-randombattle")
stat_names = ['hp','atk','def','spa','spd','spe']

In [69]:
rows = []
col_names = ['battle_id','p1_wins','elo_diff']
for i in range(6):
    for j in range(6):
        col_names.append(f"adv_M{i+1}M'{j+1}")
for file in log_dir.iterdir():
    if file.name.endswith('.json'):
        battle = b.Battle(file)
        if not (battle.custom_ruleQ) and not (battle.end_time - battle.start_time < 60): # throw out battles with custom rules and those that last less than 60 seconds
            with file.open() as battle_json:
                battle_data = json.load(battle_json)
            
            rows.append([battle_data["id"].removeprefix('gen9randombattle-'), int(battle.winner.name == battle.p1.name), battle.p1.elo0-battle.p2.elo0])
            
            team1 = [b.FullPokemon(battle_data["teams_full"][0][mon]) for mon in battle_data["teams_full"][0].keys()]
            team2 = [b.FullPokemon(battle_data["teams_full"][1][mon]) for mon in battle_data["teams_full"][1].keys()]
            for i in range(6):
                for j in range(6):
                    rows[-1].append(b.FullPokemon.advantage(team1[i],team2[j]))

df = pd.DataFrame(rows,columns=col_names)

In [71]:
df

,battle_id,p1_wins,elo_diff,adv_M1M'1,adv_M1M'2,adv_M1M'3,adv_M1M'4,adv_M1M'5,adv_M1M'6,adv_M2M'1,...,adv_M5M'3,adv_M5M'4,adv_M5M'5,adv_M5M'6,adv_M6M'1,adv_M6M'2,adv_M6M'3,adv_M6M'4,adv_M6M'5,adv_M6M'6
0,2631906096,0,-5,0.715086,1.328708,1.100367,0.649726,-0.168379,-0.605830,1.050588,...,1.039033,0.872012,0.853608,-0.258471,1.121944,1.654554,1.111303,-0.789109,0.170756,-1.030727
1,2631763570,0,10,-1.102343,-0.912040,-0.822721,0.397518,-0.967776,-1.095651,-0.166276,...,-0.498879,0.270978,-0.776190,-0.535428,0.476433,-0.809819,0.191290,0.357970,1.173952,1.035264
2,2631369343,1,-69,-1.108051,0.533011,1.163594,-0.392851,0.407047,1.024607,0.625551,...,-0.367578,0.743372,-1.148583,1.080020,1.005441,-1.007017,-0.120664,0.166639,0.305740,0.538460
3,2631529004,1,17,-0.173456,-0.971856,0.274374,-0.283311,0.510040,1.128905,-0.796007,...,0.212533,-0.519739,-1.129192,0.670411,1.063704,0.927443,0.942618,-0.156272,1.043459,0.814467
4,2631993792,0,58,-0.290351,0.339730,0.159254,-0.404944,-0.439897,-0.794229,0.800098,...,-1.295679,-0.469600,-0.618894,0.692243,1.033497,-0.499937,-0.197355,1.482827,0.203640,1.341519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4623,2631440114,1,41,0.466568,-0.332259,0.262304,1.282600,0.700100,-0.670539,-0.794315,...,0.961099,-1.040206,-0.575411,-0.390277,-0.284001,1.043605,0.934219,0.594527,1.647710,0.551990
4624,2631435014,0,-41,-0.338643,-1.068024,-0.313621,-0.768309,-0.505119,-0.287456,-0.622362,...,0.364622,-0.906603,-0.525597,0.401682,0.663443,-0.796316,0.647436,-0.761791,0.451214,-1.596547
4625,2632021860,1,119,-0.354663,1.171018,-0.593018,0.348115,0.725493,1.178656,0.703707,...,0.067488,-0.259890,0.345610,1.483343,0.576941,1.248727,-0.234713,0.226299,0.593333,-0.349652
4626,2631386449,1,-13,0.879972,0.764888,0.848133,0.908085,-0.515956,1.124624,0.894369,...,-0.296683,-0.722514,0.398275,0.357083,0.646008,0.733701,0.447504,0.663224,-0.223165,0.846311


# Random Forest things

In [124]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression

In [107]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier, ExtraTreesClassifier

In [122]:
X = df.loc[df.index < 2500].drop(['battle_id', 'p1_wins', 'elo_diff'], axis=1)
y = df.loc[df.index < 2500, ['p1_wins']]

In [123]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y)

In [117]:
DT = DecisionTreeClassifier(
    max_depth=40
)

RF = RandomForestClassifier(
    n_estimators=100,
    max_samples=0.30,
    max_features=4
)

In [118]:
DT.fit(X_train, y_train)
RF.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",4
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_tr

In [119]:
DT_avg = 0
RF_avg = 0
for i in range(100):
    
print(DT.score(X_test, y_test))
print(RF.score(X_test, y_test))

0.5
0.508


0.524